# Positive-MCP event rates in the v16 trajectory coordinates

The notebooks use the exact incoming-ray convention from the original v16 single-point workflow:

$$
\hat{\mathbf u}=(\cos\theta,\,\sin\theta\cos\alpha,\,\sin\theta\sin\alpha),
$$

$$
\hat{\mathbf b}=\cos\psi\,\hat{\mathbf e}_\theta+\sin\psi\,\hat{\mathbf e}_\alpha,
\qquad
\mathbf r_{\mathrm{in}}=b\hat{\mathbf b}-\sqrt{R^2-b^2}\,\hat{\mathbf u},
\qquad
\mathbf v_{\mathrm{in}}=v\hat{\mathbf u}.
$$

The grid now uses the **same validated coupled-Mathieu/Floquet/House pseudopotential-validity classifier in both notebooks**. The local lowest-order $q_{\max}$ and secular/RF values are retained only as diagnostics. They no longer decide whether a grid point is sampled.

The rate outputs are

$$
\dot N_{\mathrm{ph},k},\qquad
\Gamma_{\ge1,k},\qquad
\Gamma_{M,k},
$$

where $M$ is set by `exact_phonon_number` (default $M=1$). The first counts expected phonons per second, while the two $\Gamma$ quantities count passage events per second under a coherent-state/Poisson phonon-number model.

## Notebook A - fast screening

The screening estimator uses the analytic finite-frequency Coulomb pulse and a straight-line barrier test. It is intended for fast mapping and for identifying points that need the staged solver. The default absolute normalization uses the explicitly labeled `constant_local` benchmark $n_\chi=10^9\,\mathrm{m}^{-3}=10^3\,\mathrm{cm}^{-3}$. This is the PRX-Quantum benchmark ambient scale, not a derived ion density. Set `density_model="prx_two_wall"` with explicit compartment barriers to use the optional PRX Appendix-A source-density model.


The upper copper geometry is now the user-supplied rectangular copper block **minus a rectangular through-gap**. The gap is vacuum and is honored by both screening and staged structure intersections.


In [ ]:
from pathlib import Path
import sys, json, math, time, importlib.util
from dataclasses import fields
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.colors import LogNorm, TwoSlopeNorm, ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

ROOT = Path.cwd()
# Prefer the versioned convergence-certified module. This avoids accidentally
# loading an older mcp_v16_event_rate.py left in the working directory.
VERSIONED_MODULE = ROOT / "mcp_v16_event_rate_convergence.py"
LEGACY_MODULE = ROOT / "mcp_v16_event_rate.py"
if not VERSIONED_MODULE.exists() and not LEGACY_MODULE.exists():
    ROOT = Path("/mnt/data/mcp_event_rate_v16_compare")
    VERSIONED_MODULE = ROOT / "mcp_v16_event_rate_convergence.py"
    LEGACY_MODULE = ROOT / "mcp_v16_event_rate.py"

MODULE_PATH = (VERSIONED_MODULE if VERSIONED_MODULE.exists() else LEGACY_MODULE).resolve()
if not MODULE_PATH.exists():
    raise FileNotFoundError(
        "Could not find the shared event-rate module beside the notebook. "
        "Keep mcp_v16_event_rate_convergence.py (preferred) or "
        "mcp_v16_event_rate.py in the same folder as this notebook."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Always create a fresh module object from this exact path; no sys.path/cache import.
spec = importlib.util.spec_from_file_location("mcp_v16_event_rate_active", MODULE_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load {MODULE_PATH}")
mcp = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = mcp
spec.loader.exec_module(mcp)

required_functions = [
    "mathieu_validity_mesh", "dense_validity_grid", "mathieu_reference_boundaries",
    "scanconfig_audit", "hessian_convergence_report", "lower_hoa_quantum_geometry",
    "density_for_point", "ray_ground_first_entry"
]

required_scan_fields = {
    "adaptive_sampling", "min_samples_per_point", "target_mc_rel_se",
    "adaptive_check_every", "exact_phonon_number",
    "density_model", "ode_method", "ode_atol_position_m",
    "ode_atol_velocity_m_s", "ode_atol_quadrature_N_s",
}
scan_fields = {f.name for f in fields(mcp.ScanConfig)}
missing_fields = sorted(required_scan_fields - scan_fields)
missing_functions = [name for name in required_functions if not hasattr(mcp, name)]
module_version = getattr(mcp, "MODULE_VERSION", "<missing>")
if missing_fields or missing_functions or module_version < "2026-08-11.1":
    raise ImportError(
        "The shared event-rate module is outdated or mismatched.\n"
        f"Loaded: {MODULE_PATH}\n"
        f"MODULE_VERSION={module_version}\n"
        f"Missing ScanConfig fields={missing_fields}\n"
        f"Missing functions={missing_functions}\n"
        "Replace the module with mcp_v16_event_rate_convergence.py supplied "
        "with this notebook, restart the kernel, and run from the first cell."
    )

ScanConfig = mcp.ScanConfig
PRXPopulationConfig = mcp.PRXPopulationConfig
CopperGroundPlane = mcp.CopperGroundPlane
direction_basis = mcp.direction_basis
initial_state_from_v16 = mcp.initial_state_from_v16
mathieu_metrics = mcp.mathieu_metrics
mathieu_validity_mesh = mcp.mathieu_validity_mesh
dense_validity_grid = mcp.dense_validity_grid
mathieu_reference_boundaries = mcp.mathieu_reference_boundaries
run_grid = mcp.run_grid
result_matrix = mcp.result_matrix
compare_frames = mcp.compare_frames
save_config = mcp.save_config
parameter_grid = mcp.parameter_grid

scanconfig_audit = mcp.scanconfig_audit
hessian_convergence_report = mcp.hessian_convergence_report
lower_hoa_quantum_geometry = mcp.lower_hoa_quantum_geometry
density_for_point = mcp.density_for_point

OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)
print("Using module:", MODULE_PATH)
print("Module version:", mcp.MODULE_VERSION)
print("Validated v16 coupled-Floquet/House mask is active.")


## 1. Configuration

`RUN_PROFILE="production"` uses the same $36\times36$ $(m_\chi,\epsilon)$ grid as the staged notebook. Only points classified as pseudopotential-valid by the validated v16 map are evaluated. Invalid points are recorded with `effective_samples = 0` and no phase-space samples are drawn.

In [ ]:
RUN_PROFILE = "production"  # "smoke" or "production"

COMMON = dict(
    m_min_kg=1e-30, m_max_kg=1e-16,
    eps_min=1e-8, eps_max=1.0,
    density_model="constant_local", density_m3=1e9, temperature_K=300.0,
    target_mode=2, exact_phonon_number=1,
    seed=20260805,
)

if RUN_PROFILE == "smoke":
    cfg = ScanConfig(**COMMON, n_mass=24, n_eps=24, samples_per_point=32,
                     b_cap_m=2e-4, R_outer_m=20e-3,
                     delayed_branch_samples=2)
elif RUN_PROFILE == "production":
    cfg = ScanConfig(**COMMON, n_mass=36, n_eps=36, samples_per_point=2048,
                     b_cap_m=1e-3, R_outer_m=40e-3,
                     delayed_branch_samples=4)
else:
    raise ValueError(RUN_PROFILE)

ground = CopperGroundPlane(
    # User-supplied CSG geometry defaults:
    # outer block [(-12063,-10759,1376),(13413,10759,2977)] um
    # minus through-gap [(-4115,-1651,1376),(2542,1651,2977)] um
    double_layer_eV=3.19,
)
save_config(cfg, ground, OUTPUT / "screening_config.json")
print(cfg)


### Configuration and geometry audit helpers
These tables/dictionaries are generated by the shared module and mirror the corresponding sections of the documentation PDF.


In [ ]:
display(scanconfig_audit())
print("Upper copper outer bounds [m]:", ground.lo, ground.hi)
print("Upper copper gap bounds [m]:", ground.gap_lo, ground.gap_hi)
print("Lower HOA field-geometry audit:")
lower_hoa_quantum_geometry()


## 2. Validated v16 pseudopotential region

The accepted class comes from the coupled Floquet stability and House Fourier-hierarchy topology. Because those dimensionless matrices scale with $\epsilon/m_\chi$, each reference boundary is a straight line $m_\chi=C\epsilon$. The dark region below is the only region sampled by `run_grid`.

In [ ]:
def _log_edges(values):
    values = np.asarray(values, float)
    lv = np.log(values)
    edges = np.empty(values.size + 1)
    edges[1:-1] = np.exp(0.5 * (lv[:-1] + lv[1:]))
    edges[0] = np.exp(lv[0] - 0.5 * (lv[1] - lv[0]))
    edges[-1] = np.exp(lv[-1] + 0.5 * (lv[-1] - lv[-2]))
    return edges


def _draw_reference_background(ax, cfg):
    md, ed, stable, pseudo, _, _ = dense_validity_grid(cfg, n_mass=700, n_eps=700)
    classes = np.where(~stable, 0, np.where(pseudo, 2, 1)).T
    # 0 unstable (white), 1 stable/non-pseudo (gray), 2 valid (transparent-ish
    # dark background that is covered by rate colors where data exist).
    bg_cmap = ListedColormap(["white", "0.78", "0.34"])
    bg_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], bg_cmap.N)
    ax.pcolormesh(_log_edges(ed), _log_edges(md), classes, shading="flat",
                  cmap=bg_cmap, norm=bg_norm, zorder=0)

    e_line = np.geomspace(cfg.eps_min, cfg.eps_max, 1800)
    for boundary in mathieu_reference_boundaries(cfg):
        m_line = boundary["slope_kg"] * e_line
        visible = (m_line >= cfg.m_min_kg) & (m_line <= cfg.m_max_kg)
        if not np.any(visible):
            continue
        kind = boundary["kind"]
        linestyle = {"pseudopotential": ":", "stability": "--",
                     "stability+pseudopotential": "-."}[kind]
        lw = 0.9 if kind == "pseudopotential" else 1.35
        ax.plot(e_line[visible], m_line[visible], color="black",
                linestyle=linestyle, linewidth=lw, alpha=0.95, zorder=7)


def _validity_legend():
    return [
        Patch(facecolor="0.34", edgecolor="none", label="Stable; pseudopotential valid"),
        Patch(facecolor="0.78", edgecolor="none", label="Stable; pseudopotential not valid"),
        Patch(facecolor="white", edgecolor="0.4", label="Unstable"),
        Line2D([0], [0], color="black", linestyle=":", label="Pseudopotential boundary"),
        Line2D([0], [0], color="black", linestyle="--", label="Mathieu stability boundary"),
        Line2D([0], [0], color="black", linestyle="-.", label="Combined boundary"),
    ]


def plot_validity_reference(cfg, path):
    fig, ax = plt.subplots(figsize=(9.5, 6.7))
    _draw_reference_background(ax, cfg)
    ax.legend(handles=_validity_legend(), loc="lower right", fontsize=8, framealpha=0.9)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(cfg.eps_min, cfg.eps_max); ax.set_ylim(cfg.m_min_kg, cfg.m_max_kg)
    ax.set_xlabel(r"Charge fraction $\epsilon$ ($Q=\epsilon e$)")
    ax.set_ylabel(r"DM mass $m_\chi$ [kg]")
    ax.set_title("Coupled Mathieu stability and pseudopotential validity\nvalidated v16 boundary topology")
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.show()


def plot_metric(df, cfg, column, title, colorbar_label, path, contours=True):
    masses, eps, Z = result_matrix(df, column)
    Z = Z.T  # rows -> mass, columns -> epsilon for plotting
    _, _, valid = result_matrix(df, "pseudopotential_valid")
    valid = valid.T.astype(bool)
    finite = valid & np.isfinite(Z) & (Z > 0)

    fig, ax = plt.subplots(figsize=(9.5, 6.7))
    _draw_reference_background(ax, cfg)
    if finite.any():
        P = np.ma.masked_where(~finite, Z)
        vals = Z[finite]
        vmin, vmax = float(np.nanmin(vals)), float(np.nanmax(vals))
        if vmax <= vmin * (1 + 1e-12):
            vmin /= np.sqrt(10.0); vmax *= np.sqrt(10.0)
        cmap = plt.get_cmap("viridis").copy(); cmap.set_bad((0, 0, 0, 0))
        im = ax.pcolormesh(_log_edges(eps), _log_edges(masses), P, shading="flat",
                           cmap=cmap, norm=LogNorm(vmin=vmin, vmax=vmax), zorder=3)
        fig.colorbar(im, ax=ax).set_label(colorbar_label)
        if contours and np.count_nonzero(finite) >= 10:
            L = np.log10(np.where(finite, Z, np.nan))
            lo, hi = np.nanmin(L), np.nanmax(L)
            if hi - lo > 0.4:
                levels = np.unique(np.round(np.linspace(lo, hi, 5), 1))
                cs = ax.contour(eps, masses, L, levels=levels, colors="black",
                                linewidths=0.75, zorder=6)
                ax.clabel(cs, fmt=lambda x: f"$10^{{{x:.1f}}}$", fontsize=8)

    ax.legend(handles=_validity_legend()[1:], loc="lower right", fontsize=8, framealpha=0.88)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(cfg.eps_min, cfg.eps_max); ax.set_ylim(cfg.m_min_kg, cfg.m_max_kg)
    ax.set_xlabel(r"Charge fraction $\epsilon$ ($Q=\epsilon e$)")
    ax.set_ylabel(r"DM mass $m_\chi$ [kg]")
    ax.set_title(title + "\nrate shown only where the validated pseudopotential mask is valid")
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.show()


def plot_relative_error(df, cfg, column, title, path):
    masses, eps, Z = result_matrix(df, column)
    Z = Z.T
    _, _, valid = result_matrix(df, "pseudopotential_valid")
    valid = valid.T.astype(bool)
    finite = valid & np.isfinite(Z) & (Z > 0)
    fig, ax = plt.subplots(figsize=(9.5, 6.7))
    _draw_reference_background(ax, cfg)
    if finite.any():
        P = np.ma.masked_where(~finite, Z)
        cmap = plt.get_cmap("viridis").copy(); cmap.set_bad((0, 0, 0, 0))
        im = ax.pcolormesh(_log_edges(eps), _log_edges(masses), P, shading="flat",
                           cmap=cmap, norm=LogNorm(vmin=max(float(np.nanmin(Z[finite])), 1e-4),
                                                  vmax=max(float(np.nanmax(Z[finite])), 1e-3)), zorder=3)
        fig.colorbar(im, ax=ax).set_label("relative Monte Carlo standard error")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(cfg.eps_min, cfg.eps_max); ax.set_ylim(cfg.m_min_kg, cfg.m_max_kg)
    ax.set_xlabel(r"Charge fraction $\epsilon$ ($Q=\epsilon e$)")
    ax.set_ylabel(r"DM mass $m_\chi$ [kg]")
    ax.set_title(title)
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.show()

plot_validity_reference(cfg, OUTPUT / "mathieu_validity_reference.png")

masses, eps_values = parameter_grid(cfg)
stable_grid, valid_grid, _, _ = mathieu_validity_mesh(masses, eps_values, cfg)
print(f"Discrete grid: {valid_grid.sum()} / {valid_grid.size} points are pseudopotential-valid.")
for m, e in [(1e-24, 1.0), (1e-16, 1e-8), (1e-16, 1.0)]:
    print(m, e, mathieu_metrics(m, e, cfg))


## 3. Run the screening grid

For each valid point the code samples the flux-weighted Maxwell speed distribution, isotropic directions, uniform impact-plane angle $\psi$, and the corrected area/log mixture proposal for $b$. The importance denominator uses the **actual piecewise mixture density**, with $q_L(b)=0$ below $b_{\min}$, so the finite-support importance estimator is unbiased.

No phase-space sample is drawn for invalid grid points.

In [ ]:
t0 = time.time()
screening_df = run_grid("screening", cfg, ground, progress=True)
print(f"Runtime: {time.time() - t0:.2f} s")
screening_df.to_csv(OUTPUT / "screening_event_rate.csv", index=False)

invalid = ~screening_df["pseudopotential_valid"].astype(bool)
assert (screening_df.loc[invalid, "effective_samples"] == 0).all()
assert screening_df.loc[invalid, ["phonon_rate_s", "event_rate_ge1_s", "event_rate_exact_M_s"]].isna().all().all()
print(f"Verified: {invalid.sum()} invalid points received zero samples.")
screening_df.head()


## 4. Rate maps

Three maps are produced on the identical valid region:

- $\dot N_{\mathrm{ph},k}$: expected phonons per second;
- $\Gamma_{\ge1,k}$: rate of passages producing at least one phonon;
- $\Gamma_{M,k}$: rate of passages producing exactly $M$ phonons, with $M=$ `cfg.exact_phonon_number`.

The distinction matters when a single close passage produces many phonons: $\dot N_{\mathrm{ph},k}$ can greatly exceed $\Gamma_{\ge1,k}$.

In [ ]:
plot_metric(screening_df, cfg, "phonon_rate_s",
            "Fast screening: selected-mode phonon production rate",
            r"$\dot N_{\mathrm{ph},k}$ [s$^{-1}$]",
            OUTPUT / "screening_phonon_rate.png")
# Backward-compatible filename used by earlier packages.
plot_metric(screening_df, cfg, "phonon_rate_s",
            "Fast screening: selected-mode phonon production rate",
            r"$\dot N_{\mathrm{ph},k}$ [s$^{-1}$]",
            OUTPUT / "screening_heatmap.png")
plot_metric(screening_df, cfg, "event_rate_ge1_s",
            "Fast screening: at-least-one-phonon event rate",
            r"$\Gamma_{\geq1,k}$ [s$^{-1}$]",
            OUTPUT / "screening_gamma_ge1.png")
plot_metric(screening_df, cfg, "event_rate_exact_M_s",
            f"Fast screening: exact-{cfg.exact_phonon_number}-phonon event rate",
            rf"$\Gamma_{{{cfg.exact_phonon_number},k}}$ [s$^{{-1}}$]",
            OUTPUT / f"screening_gamma_M{cfg.exact_phonon_number}.png")


## 5. Comparison hook

After the staged notebook finishes, rerun this cell to generate the merged CSV. The staged notebook produces the three corresponding ratio maps.

In [ ]:
staged_path = OUTPUT / "staged_event_rate.csv"
if staged_path.exists():
    staged_df = pd.read_csv(staged_path)
    comparison = compare_frames(staged_df, screening_df)
    comparison.to_csv(OUTPUT / "staged_vs_screening.csv", index=False)
    display(comparison.head())
else:
    print("Run MCP_Event_Rate_Full_Staged_v16_Coordinates.ipynb first.")
